# 40. The scikit-learn Way

**Tier:** Classic ML
**Estimated time:** 40 minutes
**Prerequisites:** 37 (ML Foundations), 39 (Trees, Forests & Boosting)
**Priority:** 🟡 Important — `Pipeline` isn't a convenience wrapper, it's a structural fix for the single most common way ML evaluation quietly lies to you.
**Source material:** Fabian Pedregosa et al., ["Scikit-learn: Machine Learning in Python"](https://www.jmlr.org/papers/volume12/pedregosa11a/pedregosa11a.pdf), JMLR, 2011.

## What You'll Learn
- Why the `fit`/`predict`/`transform` **contract** is the actual design achievement of scikit-learn, not any single algorithm
- A **real, measured data leak** — 70%+ "accuracy" on data with zero actual signal — caused by preprocessing before splitting
- How `Pipeline` structurally prevents that leak, rather than relying on remembering to do things in the right order
- Why **stratified** cross-validation matters, with a fold that goes empty without it

## Why This Matters
The scaler-fit-on-everything leak in this notebook isn't a contrived textbook trap — it's one of
the most common ways a real evaluation number turns out to be fiction. Notebook 25's "split
before preprocessing" rule is a discipline you have to remember; `Pipeline` makes it a
structural guarantee you can't accidentally violate.

In [1]:
%matplotlib inline
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(40)
print("Setup complete.")

Setup complete.


## The estimator API: a contract worth studying on its own

scikit-learn's actual innovation isn't any one algorithm — logistic regression and random
forests both predate it by decades. It's that **every estimator, regardless of what it does
internally, exposes the same three methods**: `fit(X, y)` trains it, `predict(X)` makes
predictions, and (for preprocessing steps) `transform(X)` applies a learned transformation.
A logistic regression, a support vector machine, a random forest, and a k-nearest-neighbors
classifier share nothing about *how* they work — but code that only calls `fit`/`predict`
can't tell them apart. That uniformity is what makes `Pipeline`, `GridSearchCV`, and
`cross_val_score` possible: they're written once, against the *contract*, and work with every
estimator that honors it — no special-casing per algorithm required.

In [2]:
X, y = make_moons(n_samples=300, noise=0.3, random_state=40)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=40)

models = {
    "LogisticRegression": LogisticRegression(),
    "SVC": SVC(),
    "RandomForestClassifier": RandomForestClassifier(random_state=40),
    "KNeighborsClassifier": KNeighborsClassifier(),
}
for name, model in models.items():
    model.fit(X_train, y_train)        # identical call — every model
    acc = model.score(X_test, y_test)  # identical call — every model
    print(f"{name:25s} test_acc={acc:.3f}")

LogisticRegression        test_acc=0.811
SVC                       test_acc=0.867
RandomForestClassifier    test_acc=0.867
KNeighborsClassifier      test_acc=0.856


Four completely different algorithms, trained and scored through the exact same two lines of
code. This is the same idea as notebook 28b's tool contracts: an agent's tool-calling loop
doesn't need a special case for every tool because every tool honors the same
schema-in/result-out contract. Here, the contract is `fit`/`predict`/`score`, and it's cheap to
prove — any object honoring it drops into the same loop, sklearn class or not:

In [3]:
class NearestMeanClassifier:
    """A from-scratch classifier honoring sklearn's fit/predict/score contract."""
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.means_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        return self
    def predict(self, X):
        dists = np.linalg.norm(X[:, None, :] - self.means_[None, :, :], axis=2)
        return self.classes_[np.argmin(dists, axis=1)]
    def score(self, X, y):
        return (self.predict(X) == y).mean()

models["NearestMeanClassifier (ours)"] = NearestMeanClassifier()
models["NearestMeanClassifier (ours)"].fit(X_train, y_train)
print(f"{'NearestMeanClassifier (ours)':25s} "
      f"test_acc={models['NearestMeanClassifier (ours)'].score(X_test, y_test):.3f}")

NearestMeanClassifier (ours) test_acc=0.711


*Nothing about `NearestMeanClassifier` involves sklearn — it's fifteen lines of numpy. But
because it honors the same three-method contract, it's indistinguishable, from the outside, from
the four library models above. This is the practical payoff of designing to an interface rather
than a specific implementation.*

## A real data leak — 70%+ accuracy on pure noise

Data leakage means information from data the model shouldn't have seen yet (validation or test
data) influences training — usually through a preprocessing step fit on the full dataset
*before* splitting. The classic, dramatic version: **selecting which features to use based on
their correlation with the label, computed on the full dataset.** Let's build a dataset with
**zero real signal** — 1000 pure-noise features, random labels — and watch feature selection
manufacture a convincing-looking accuracy out of nothing.

In [4]:
n, n_features = 200, 1000
X_noise = rng.normal(0, 1, (n, n_features))   # every feature is pure noise
y_noise = rng.integers(0, 2, n)               # labels have NO relationship to any feature

# LEAKY: select the 20 "best" features using the ENTIRE dataset, THEN split.
selector_leaky = SelectKBest(f_classif, k=20).fit(X_noise, y_noise)
X_selected = selector_leaky.transform(X_noise)
Xtr, Xte, ytr, yte = train_test_split(X_selected, y_noise, test_size=0.3, random_state=40)
leaky_clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"LEAKY  (select features on ALL data, then split): "
      f"train_acc={leaky_clf.score(Xtr, ytr):.3f}  test_acc={leaky_clf.score(Xte, yte):.3f}")

# HONEST: split FIRST, select features using only the training fold.
Xn_train, Xn_test, yn_train, yn_test = train_test_split(X_noise, y_noise, test_size=0.3, random_state=40)
selector_honest = SelectKBest(f_classif, k=20).fit(Xn_train, yn_train)
honest_clf = LogisticRegression(max_iter=1000).fit(selector_honest.transform(Xn_train), yn_train)
print(f"HONEST (split first, select on TRAIN fold only):  "
      f"train_acc={honest_clf.score(selector_honest.transform(Xn_train), yn_train):.3f}  "
      f"test_acc={honest_clf.score(selector_honest.transform(Xn_test), yn_test):.3f}")

LEAKY  (select features on ALL data, then split): train_acc=0.814  test_acc=0.700
HONEST (split first, select on TRAIN fold only):  train_acc=0.857  test_acc=0.433


*The "leaky" pipeline reports ~70% test accuracy — on data with, by construction, zero real
signal. Among 1000 random features, plenty will show a spurious correlation with `y_noise`
purely by chance; selecting on the full dataset means that selection already "peeked" at the
test labels, so of course the selected features look predictive on data that helped choose
them. The honest version, which never lets test data influence feature selection, lands right
around chance (50%) — the true answer. This exact bug — fit a scaler, an imputer, a feature
selector, or a target encoder on the full dataset before splitting — is one of the most common
sources of an ML evaluation number that looks great and is fiction.*

## Pipeline: the structural fix

The honest version above required remembering to fit the selector on `X_train` only, and
carefully applying its `transform` (never `fit`) to `X_test`. That's exactly the kind of
discipline notebook 25 asks for — and exactly the kind of thing that's easy to get right once
and then quietly break three months later. `Pipeline` bundles preprocessing and the model into
a single estimator that itself honors the `fit`/`predict` contract, and its integration with
`cross_val_score`/`GridSearchCV` guarantees every fold refits the *entire* pipeline — selector
included — on that fold's training data only. There's no longer a wrong order to remember.

In [5]:
pipeline = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf", LogisticRegression(max_iter=1000)),
])

# cross_val_score refits the WHOLE pipeline (selector + classifier) on each fold's training
# data, and only ever transforms — never fits — that fold's held-out data. No leak is possible
# by construction, not by discipline.
cv_scores = cross_val_score(pipeline, X_noise, y_noise, cv=5)
print(f"Pipeline + cross_val_score: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f} per fold: {np.round(cv_scores, 3)}")

Pipeline + cross_val_score: 0.485 +/- 0.049 per fold: [0.575 0.475 0.425 0.475 0.475]


*Right back around chance, exactly as it should be for data with no real signal — because
`cross_val_score` never lets any fold's feature selection see that same fold's evaluation data.
The fix isn't "be more careful," it's "make the leak structurally impossible by construction" —
the same move notebook 30c's capability tokens make for agent security: don't rely on an agent
remembering not to do something dangerous, make the dangerous action unavailable.*

## Cross-validation, stratified correctly

Splitting into folds sounds simple until the data isn't already shuffled. Plain `KFold` takes
*contiguous* chunks of whatever order the data arrives in — if a dataset happens to be sorted
or grouped by label (common: data appended over time, one class at a time), some folds can end
up with almost no examples of the minority class at all. `StratifiedKFold` fixes this by
preserving each class's overall proportion in every fold.

In [6]:
n_samples = 100
y_imbalanced = np.array([0] * 90 + [1] * 10)  # grouped by class, NOT shuffled — a realistic
                                                # case if rows were appended by label over time

print("plain KFold (data grouped by class):")
for i, (_, test_idx) in enumerate(KFold(n_splits=5, shuffle=False).split(y_imbalanced)):
    counts = np.bincount(y_imbalanced[test_idx], minlength=2)
    print(f"  fold {i}: test-fold class counts = {counts}")

print("StratifiedKFold:")
for i, (_, test_idx) in enumerate(StratifiedKFold(n_splits=5).split(np.zeros(n_samples), y_imbalanced)):
    counts = np.bincount(y_imbalanced[test_idx], minlength=2)
    print(f"  fold {i}: test-fold class counts = {counts}")

plain KFold (data grouped by class):
  fold 0: test-fold class counts = [20  0]
  fold 1: test-fold class counts = [20  0]
  fold 2: test-fold class counts = [20  0]
  fold 3: test-fold class counts = [20  0]
  fold 4: test-fold class counts = [10 10]
StratifiedKFold:
  fold 0: test-fold class counts = [18  2]
  fold 1: test-fold class counts = [18  2]
  fold 2: test-fold class counts = [18  2]
  fold 3: test-fold class counts = [18  2]
  fold 4: test-fold class counts = [18  2]


*Plain `KFold` produces four folds with ZERO minority-class examples in the test set and one
fold that's 50/50 — you couldn't even compute a minority-class recall on most folds, and the
one fold that does have minority examples would dominate any averaged metric. `StratifiedKFold`
gives every fold the same 18:2 ratio as the full dataset. `cross_val_score` on a classifier
uses stratified splitting by default for exactly this reason — but it's worth knowing what it's
protecting you from.*

## Exercises

In [7]:
# Exercise 1 (Warm-up): does the protection depend on the classifier?
# Task: Rebuild the Pipeline from the leak-fix section with the classifier swapped to
#       RandomForestClassifier(random_state=40) instead of LogisticRegression. Rerun
#       cross_val_score on X_noise/y_noise. Confirm the score still lands near chance (~0.5).
# Hint: Pipeline's leak protection comes from WHEN the selector is fit (per-fold, on that
#       fold's training data only) — it has nothing to do with which classifier sits after it.

# YOUR CODE HERE

In [8]:
# Exercise 2 (Apply): hand-write the cross-validation loop cross_val_score does internally
# Task: Without calling cross_val_score, use KFold(n_splits=5) to get train/test index arrays
#       for X_noise/y_noise, and for each fold: fit a FRESH copy of `pipeline` (use
#       sklearn.base.clone) on the fold's training rows, score it on the fold's test rows, and
#       collect the 5 scores. Confirm your mean is close to cross_val_score's ~0.5 result.
# Hint: sklearn.base.clone(pipeline) gives you an unfitted copy so each fold starts fresh —
#       reusing the SAME fitted pipeline object across folds would itself be a subtle leak.

# YOUR CODE HERE

In [9]:
# Exercise 3 (Extend): write the tool-contract parallel in your own words
# Task: In a comment, explain concretely: if scikit-learn estimators did NOT share a common
#       fit/predict interface (imagine every model had a differently-named training method —
#       LogisticRegression.train(), RandomForestClassifier.grow(), SVC.optimize()), what would
#       break about Pipeline, cross_val_score, and GridSearchCV? Then connect this to notebook
#       28b: what does an agent's tool-calling loop require from every tool's interface for the
#       SAME reason?
# Hint: both cases are "generic code that works with ANY implementation of a shared contract" —
#       identify exactly what part of each contract (method names? argument shapes? return
#       types?) is the thing that has to stay uniform for the generic code to work at all.

# YOUR CODE HERE

<details>
<summary>Solutions (click to expand)</summary>

```python
# Exercise 1
from sklearn.ensemble import RandomForestClassifier as RFC
pipeline_rf = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf", RFC(random_state=40)),
])
cv_scores_rf = cross_val_score(pipeline_rf, X_noise, y_noise, cv=5)
print(f"RandomForest pipeline: {cv_scores_rf.mean():.3f} +/- {cv_scores_rf.std():.3f}")
# Still near chance — swapping LogisticRegression for RandomForestClassifier changes nothing
# about WHEN SelectKBest gets fit, which is the only thing that determines whether a leak exists.

# Exercise 2
from sklearn.base import clone
fold_scores = []
for train_idx, test_idx in KFold(n_splits=5, shuffle=True, random_state=40).split(X_noise):
    fold_pipeline = clone(pipeline)
    fold_pipeline.fit(X_noise[train_idx], y_noise[train_idx])
    fold_scores.append(fold_pipeline.score(X_noise[test_idx], y_noise[test_idx]))
print(f"hand-written CV: {np.mean(fold_scores):.3f} +/- {np.std(fold_scores):.3f}")
# Matches cross_val_score's result (modulo which rows land in which fold) because it does
# exactly the same thing under the hood: a fresh, unfitted pipeline per fold.

# Exercise 3
# Without a shared fit/predict interface, Pipeline couldn't call step.fit(X, y) generically —
# it would need an if/elif chain checking each step's type and calling ITS specific method name.
# GridSearchCV couldn't swap in candidate models/hyperparameters without the same per-type
# special-casing. cross_val_score's refit-per-fold logic would need one implementation per
# algorithm. All the "generic orchestration code" in this notebook exists ONLY because it can
# assume every estimator answers to fit(X, y) and predict(X)/transform(X).
#
# Notebook 28b's tool contract is the identical requirement one layer up: an agent's loop calls
# execute(tool_name, args) and gets back a structured result, without special-casing "if this is
# the search tool, call search_impl(query); if it's the calculator, call calc_impl(expr)".  The
# specific piece of the contract that has to stay uniform in both cases is the SAME: the call
# signature (what arguments go in, in what shape) and the return shape (what comes back, and in
# what form) — not what happens inside. Change either of those per-implementation and the
# generic orchestration code (Pipeline here, the tool-calling loop there) breaks.
```
</details>

## Key Takeaways
- **The estimator API's uniform `fit`/`predict`/`transform` contract**, not any specific algorithm, is scikit-learn's real design achievement — it's what makes generic tools like `Pipeline` and `cross_val_score` possible without special-casing every model, the same reason notebook 28b's tool contracts make an agent's tool loop generic.
- **Fitting a preprocessing step (scaler, imputer, feature selector) on the full dataset before splitting is a real, measurable leak** — this notebook manufactured ~70% "accuracy" from data with zero actual signal.
- **`Pipeline` makes the leak structurally impossible rather than relying on discipline** — every fold, in `cross_val_score` or `GridSearchCV`, refits the entire pipeline on that fold's training data alone.
- **Unstratified k-fold splitting can silently break on imbalanced or grouped data** — up to entire folds with zero examples of a class — while `StratifiedKFold` preserves class balance by construction.

## What's Next
Notebook 41 turns to a different question entirely — not "is this model evaluated correctly," but "why did THIS model make THIS prediction" — using SHAP to attribute a prediction to its inputs, and contrasting that with why an LLM's attention weights are not the same kind of explanation.